# pandas: таблица поездок каршеринга

Отдел аналитики «Дорога» хранит поездки в CSV. В модуле 1 `predict` работал на парах чисел; здесь **сотни строк** и **несколько полей** на поездку — нужен `DataFrame`.

In [ ]:
from pathlib import Path
import pandas as pd


def find_trips_csv() -> Path:
    for p in (Path('trips.csv'), Path('../../data/trips.csv'), Path('../data/trips.csv')):
        if p.exists():
            return p.resolve()
    raise FileNotFoundError('trips.csv не найден')


TRIPS_PATH = find_trips_csv()
df = pd.read_csv(TRIPS_PATH)


## 1. Зачем таблица

Одна **строка** = одна поездка; один **столбец** = одно поле (расстояние, час, зона…). Выведите число строк и столбцов, первые 5 строк и список имён столбцов.

**Вопрос:** сколько поездок в файле и сколько полей у каждой?

**Pitfall:** `print(df)` на больших данных заливает экран — для осмотра хватает `shape`, `head`, `columns`.

In [ ]:
n_rows, n_cols = None, None  # df.shape
# print(df.head())
# print(list(df.columns))
assert n_rows is not None and n_cols is not None
assert isinstance(n_rows, (int, float)) and n_rows > 50
assert n_cols >= 5
print(n_rows, n_cols)

## 2. dtypes: число vs текст

Вызовите `df.dtypes` (или `df.info()`). Числовые столбцы нужны для `fit`; `object` — строки/категории, их нельзя сразу подать в `LinearRegression`.

**How:** `df['distance_km'].dtype` vs `df['zone'].dtype`.

**Checkpoint:** почему `distance_km` не `object`?

In [ ]:
# print(df.dtypes)
distance_dtype = None  # str(df['distance_km'].dtype)
zone_dtype = None  # str(df['zone'].dtype)
assert distance_dtype is not None and zone_dtype is not None
assert 'object' not in str(distance_dtype).lower() or 'float' in str(distance_dtype).lower() or 'int' in str(distance_dtype).lower()
assert 'object' in str(zone_dtype).lower() or str(zone_dtype) == 'object'
print('distance:', distance_dtype, '| zone:', zone_dtype)

## 3. Одна ячейка через iloc

Расстояние первой поездки: `df.iloc[0]['distance_km']` (или `df.iloc[0, …]`). Запишите в `first_distance`.

**Зачем:** модель потом читает признаки **построчно**.

**Pitfall:** `iloc` — по **позиции** (0, 1, 2…), не по `trip_id`.

In [ ]:
first_distance = None
assert first_distance is not None
assert isinstance(first_distance, (int, float)) and first_distance > 0
print('первая поездка, км:', first_distance)

## 4. loc vs iloc — нюанс

Возьмите первые 3 строки столбцов `distance_km` и `duration_min` двумя способами: `.loc` (имена) и `.iloc` (позиции). Значения должны совпасть.

**Why:** после фильтра индекс строк может быть не 0…n; `iloc[0]` — «первая видимая», `loc[0]` — «строка с меткой 0» (может отсутствовать).

In [ ]:
by_name = None  # .loc
by_pos = None  # .iloc
assert by_name is not None and by_pos is not None
assert by_name.shape == (3, 2) and by_pos.shape == (3, 2)
assert list(by_name.columns) == ['distance_km', 'duration_min']
assert (by_name.values == by_pos.values).all()
print(by_name)
print(by_pos)

## 5. Признак, цель, служебный столбец

Для ML явно назовите роли:
- **цель** — что предсказываем (`duration_min`);
- **признаки** — числа/категории для модели;
- **id** — служебный ключ, **не** признак.

Заполните переменные ниже.

In [ ]:
TARGET = ''
FEATURES = []
ID_COL = ''

assert TARGET in df.columns
assert TARGET not in FEATURES
assert ID_COL in df.columns
assert ID_COL not in FEATURES
assert 'distance_km' in FEATURES
assert set(FEATURES).issubset(df.columns)
assert df[TARGET].dtype.kind in 'iuf'

## 6. value_counts: зоны

Сколько поездок в каждой `zone`? Используйте `value_counts()` (частоты категорий — база для EDA и фильтров).

Запишите число **уникальных** зон в `n_zones`.

In [ ]:
zone_counts = None  # df['zone'].value_counts()
# print(zone_counts)
n_zones = None
assert zone_counts is not None
assert n_zones is not None
assert isinstance(n_zones, (int, float)) and 2 <= n_zones <= 20
print('уникальных zone:', n_zones)
print(zone_counts)

## 7. Checkpoint: почему не list of dicts

В модуле 1 данные жили в списках. Здесь — таблица.

Напишите 2–3 предложения в `WHY_DATAFRAME`: зачем `DataFrame` вместо `list[dict]` для сотен поездок (фильтр, столбец целиком, dtypes, готовность к `sklearn`).

**How:** подумайте про `df['distance_km']` vs цикл по словарям.

In [ ]:
WHY_DATAFRAME = ''
assert len(WHY_DATAFRAME) > 40
assert 'list' in WHY_DATAFRAME.lower() or 'слов' in WHY_DATAFRAME.lower() or 'dataframe' in WHY_DATAFRAME.lower() or 'таблиц' in WHY_DATAFRAME.lower()
print(WHY_DATAFRAME)

## 8. Расширение: поездка по id

Найдите длительность поездки `T0001` через `.loc` и маску по `trip_id`.

**Зачем:** id нужен для поиска строки, но в `X` для `fit` его не кладут.

In [ ]:
duration_t0001 = None
assert duration_t0001 is not None
assert isinstance(duration_t0001, (int, float)) and duration_t0001 > 0
print('T0001 duration:', duration_t0001)